# Магістерська кваліфікаційна робота
## Тема: «Інтелектуальна система оцінки ризиків захворювань на основі методів пояснюваного штучного інтелекту»
### Науково-дослідний розділ 2: Порівняльний аналіз та бенчмаркінг методів Explainable AI (XAI)

* **Здобувач вищої освіти:** Олег Стефанів  
* **Спеціальність:** 122 «Комп’ютерні науки»  
* **Факультет:** Факультет інформаційних технологій, ДВНЗ «Ужгородський національний університет»  
* **Науковий керівник:** к.ф.-м.н., доцент Ніколенко В. В.  

---

### Актуальність та мета дослідження:
Впровадження систем підтримки прийняття клінічних рішень (CDSS) на базі алгоритмів машинного навчання стримується проблемою «чорної скриньки» (Black-Box Problem). Лікар не має права приймати життєво важливі терапевтичні рішення без розуміння причинно-наслідкової логіки алгоритму.

**Мета даного розділу:**
1. Здійснити строге порівняння провідних парадигм Explainable AI:
   * **TreeSHAP** (кооперативна теорія ігор, точний оптимізований алгоритм для ансамблів дерев);
   * **LIME** (локальні сурогатні лінійні моделі на базі пертурбацій);
   * **KernelSHAP** (модель-агностична апроксимація значень Шеплі);
   * **Explainable Boosting Machine (EBM)** (іманентно інтерпретована скляна скринька класу GA2M);
   * **Logistic Regression** (класичний лінійний інтерпретований базис).
2. Оцінити методи за **4 формальними науковими критеріями**:
   * **Швидкодія та латентність (Latency)** — час надання пояснення одному пацієнту (мс);
   * **Вірність та адекватність (Faithfulness / Feature Deletion-AUC)** — здатність точно відображати справжні драйвери ризику моделі;
   * **Математична стійкість (Robustness / Local Lipschitz)** — стабільність пояснень до стохастичного шуму вхідних біомаркерів;
   * **Міжметодна узгодженість (Inter-Method Consensus)** — рангова кореляція Спірмена та індекси Жаккара для Топ-3 / Топ-5 факторів.
3. Дослідити фундаментальний компроміс між предиктивною точністю (Accuracy) та інтерпретованістю (Interpretability).

In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display, Markdown

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11

reports_dir = os.path.join('..', 'reports')
results_json_path = os.path.join(reports_dir, 'xai_benchmark_results.json')

print("Середовище ініціалізовано. Завантаження результатів наукового бенчмаркінгу...")

## 1. Теоретичний базис методів пояснюваного штучного інтелекту

### 1.1 Метод значень Шеплі та алгоритм TreeSHAP
Метод SHAP (Shapley Additive Explanations) базується на класичній кооперативній теорії ігор Ллойда Шеплі (Нобелівська премія 2012 р.). Атрибуція $\phi_i$ фактора $i$ обчислюється як його середньозважений маржинальний внесок у всі можливі коаліції ознак $S$:

$$\phi_i(v) = \sum_{S \subseteq N \setminus \{i\}} \frac{|S|!(|N| - |S| - 1)!}{|N|!} \left[ v(S \cup \{i\}) - v(S) \right]$$

SHAP є **єдиним** методом локальної інтерпретації, який одночасно задовольняє 4 фундаментальні аксіоми:
1. **Ефективність (Efficiency):** $\sum_{i=1}^M \phi_i(x) = f(x) - E[f(x)]$.
2. **Симетрія (Symmetry):** Якщо дві ознаки вносять однаковий вклад у будь-яку коаліцію, їхні значення Шеплі рівні.
3. **Фіктивний гравець / Нульовий внесок (Dummy Player):** Якщо фактор не змінює результат жодної коаліції, $\phi_i = 0$.
4. **Адитивність (Additivity):** Для суми моделей значення Шеплі сумуються: $\phi_i(f_1 + f_2) = \phi_i(f_1) + \phi_i(f_2)$.

Алгоритм **TreeSHAP** (Lundberg et al., Nature Machine Intelligence, 2020) знижує експоненційну складність $\mathcal{O}(M 2^M)$ до поліноміальної $\mathcal{O}(TLD^2)$, де $T$ — кількість дерев, $L$ — максимальна кількість листків, $D$ — глибина дерева.

---

### 1.2 Метод локальних сурогатів LIME (Local Interpretable Model-agnostic Explanations)
LIME (Ribeiro et al., KDD 2016) апроксимує складну нелінійну функцію $f(x)$ у малому локальному околі пацієнта $x$ за допомогою простої інтерпретованої моделі $g \in G$ (звичайної гребеневої регресії Ridge):

$$\xi(x) = \arg\min_{g \in G} \mathcal{L}(f, g, \pi_x) + \Omega(g)$$

де $\pi_x(z) = \exp(-D(x,z)^2 / \sigma^2)$ — експоненційне ядро відстані, що надає найбільшу вагу пертурбаціям, близьким до пацієнта $x$, а $\Omega(g)$ — регуляризатор складності моделі пояснення.

---

### 1.3 Узагальнені адитивні моделі зі взаємодіями EBM (Explainable Boosting Machine)
EBM належить до класу моделей **GA2M** (Generalized Additive Models with Pairwise Interactions):

$$g(E[y]) = \beta_0 + \sum_{i=1}^M f_i(x_i) + \sum_{i \neq j} f_{ij}(x_i, x_j)$$

Кожна функція $f_i(x_i)$ навчається циклічним градієнтним бустингом дерев глибини 1 (stumps), що виключає вплив сторонніх факторів і забезпечує 100% прозорість без потреби в зовнішньому post-hoc експлейнері.

In [2]:
with open(results_json_path, 'r', encoding='utf-8') as f:
    benchmark_data = json.load(f)

print("Метадані експерименту:")
for k, v in benchmark_data['metadata'].items():
    print(f"  • {k}: {v}")

## 2. Порівняння предиктивної якості моделей (Accuracy vs. Interpretability)

У кардіології критично важливим є баланс між діагностичною чутливістю та інтерпретованістю моделі. Нижче наведено результати тестування на когорті CDC BRFSS ($N=50,736$):

In [3]:
perf_df = pd.DataFrame(benchmark_data['model_performance']).T
perf_df.columns = ['ROC-AUC', 'PR-AUC', 'Accuracy (пор. 25%)', 'F1-Score', 'Brier Score']
display(perf_df)

### Висновки щодо якості моделей:
1. **XGBoost (Black Box)** досягає найвищої прогностичної сили (**ROC-AUC: 84.79%**, PR-AUC: 0.3640), оскільки ефективно вловлює складні нелінійні поєднання ризиків.
2. **EBM (Glassbox)** демонструє практично ідентичну якість (**ROC-AUC: 84.67%**, відставання лише на 0.12%), будучи при цьому повністю внутрішньо прозорою моделлю.
3. **Логістична регресія** дещо поступається нелінійним моделям (ROC-AUC: 84.51%), демонструючи обмеження строго лінійного припущення.

## 3. Обчислювальна складність та швидкодія (Latency Benchmark)

Для клінічної системи реального часу вирішальне значення має час розрахунку пояснення на одного пацієнта:

In [4]:
lat_df = pd.DataFrame(list(benchmark_data['latency_ms_per_sample'].items()), columns=['Метод XAI', 'Час на 1 пацієнта (мс)'])
lat_df['Прискорення відносно LIME'] = (benchmark_data['latency_ms_per_sample']['LIME'] / lat_df['Час на 1 пацієнта (мс)']).round(1)
display(lat_df)

display(Image(filename=os.path.join(reports_dir, 'xai_latency_benchmark.png')))

## 4. Оцінка адекватності та вірності пояснень (Explanation Faithfulness)

Метод послідовного видалення ознак (**Feature Ablation Deletion-AUC**):
Якщо метод коректно визначає справжні фактори ризику, то заміна топ-$k$ виявлених ознак на базове популяційне значення повинна спричиняти максимально швидке падіння впевненості моделі. Чим більша площа під кривою зміни ризику (Deletion-AUC), тим вища вірність методу.

In [5]:
faith_df = pd.DataFrame(list(benchmark_data['faithfulness_deletion_auc'].items()), columns=['Метод XAI', 'Deletion-AUC (Вірність)'])
display(faith_df)

display(Image(filename=os.path.join(reports_dir, 'xai_faithfulness_ablation.png')))

## 5. Математична стійкість та робастність до збурень (Robustness Benchmark)

Оцінка поведінки методів при додаванні до показників пацієнта мікрошуму ($\varepsilon \sim \mathcal{N}(0, 0.02^2)$):
* **Локальна константа Ліпшиця** $\mathcal{L} = \frac{\|\phi(x) - \phi(x')\|_2}{\|x - x'\|_2}$ (менше значення — вища стабільність);
* **Рангова стійкість Спірмена** $\rho$ між чистими та шумними векторами атрибуцій.

In [6]:
stab_data = benchmark_data['stability_metrics']
stab_df = pd.DataFrame([
    {'Метод': 'TreeSHAP', 'Константа Ліпшиця': stab_data['treeshap_lipschitz'], 'Рангова стійкість (ρ)': stab_data['treeshap_rank_stability_rho']},
    {'Метод': 'LIME', 'Константа Ліпшиця': stab_data['lime_lipschitz'], 'Рангова стійкість (ρ)': stab_data['lime_rank_stability_rho']}
])
display(stab_df)

display(Image(filename=os.path.join(reports_dir, 'xai_stability_robustness.png')))

## 6. Міжметодна узгодженість (Inter-Method Consensus)

Чи сходяться TreeSHAP та LIME у визначенні ключових клінічних загроз для одного і того ж пацієнта?

In [7]:
agree_data = benchmark_data['inter_method_agreement']
agree_df = pd.DataFrame([
    {'Метрика консенсусу': 'Середня рангова кореляція Спірмена (ρ)', 'Значення': f"{agree_data['mean_spearman_rho']:.4f}"},
    {'Метрика консенсусу': 'Індекс Жаккара для Топ-3 факторів ризику', 'Значення': f"{agree_data['top3_jaccard_similarity']*100:.1f}%"},
    {'Метрика консенсусу': 'Індекс Жаккара для Топ-5 факторів ризику', 'Значення': f"{agree_data['top5_jaccard_similarity']*100:.1f}%"}
])
display(agree_df)

display(Image(filename=os.path.join(reports_dir, 'xai_agreement_correlation.png')))

## 7. Загальний компроміс «Точність — Інтерпретованість» (Accuracy vs. Interpretability)

In [8]:
display(Image(filename=os.path.join(reports_dir, 'xai_accuracy_vs_interpretability.png')))

## 8. Наукові висновки для Розділу 2 магістерської дисертації

1. **Обґрунтування вибору TreeSHAP як ядра клінічної системи:**
   * **Швидкодія:** TreeSHAP забезпечує обчислення значень Шеплі за **~0.3–0.5 мс/пацієнт**, що у **300–400 разів швидше за LIME** і у **1500+ разів швидше за KernelSHAP**. Це єдиний ігровий метод, здатний працювати в інтерактивному режимі веб-інтерфейсу.
   * **Вірність (Faithfulness):** За метрикою Deletion-AUC TreeSHAP демонструє найвищу коректність відображення внутрішньої логіки ансамблю дерев, суттєво перевершуючи локальні сурогати LIME.
   * **Математична детермінованість:** На відміну від LIME, чиї пояснення мають стохастичну дисперсію через вибірку Монте-Карло, TreeSHAP повертає строго детерміновані значення, інваріантні до перестановки вибірки.
2. **Роль скляних скриньок (EBM):**
   * Модель EBM продемонструвала вражаючу точність (ROC-AUC: 84.67%), що підтверджує перспективність використання адитивних сплайн-моделей як внутрішньо інтерпретованого компаратора.
3. **Практична цінність результатів:**
   * Розроблений стенд кількісного бенчмаркінгу XAI усуває суб'єктивність при виборі методів інтерпретації та забезпечує доказову наукову базу для захисту магістерської роботи за спеціальністю 122 «Комп'ютерні науки».